In [ ]:
import os, numpy as np
import matplotlib.pyplot as plt

from notebooks import nutils
from core import config
config.Config.configure_AMIDE_clinical_whole_brain()

import analyze_uncertainty as au

In [ ]:
_maindir = '/home/alexf/phiVAE/figs/humans/'
train_times = []
for folder in os.listdir(_maindir):
    folder_path = os.path.join(_maindir, folder)
    #if not os.path.isfile(os.path.join(folder_path, 'intermediates/MT_UQ_stats_allpoints.npz')):  # MT_UQ_stats_subset_of_points
    if not os.path.isfile(os.path.join(folder_path, 'intermediates/amide_tissue_params_est.npz')): 
        continue
    train_times += [np.load(f'{folder_path}/intermediates/amide_tissue_params_est.npz')['train_time']]
print(rf'{np.mean(train_times):.2f}\\pm {np.std(train_times):.2f} sec')
print(rf'{np.mean(train_times)/60:.1f}\\pm {np.std(train_times)/60:.1f} min')

In [ ]:
train_suffixes = ['vol8-9-10_2026-04-28_h20-m57', 'vol7-9-10_2026-04-28_h22-m45', 'vol7-8-10_2026-04-30_h14-m34', 'vol7-8-9_2026-04-29_h00-m35']

In [ ]:

vol_sli = -30 # 15 20

trainfolders = [f'/home/alexf/phiVAE/ckpts/healthy/{folder}' for folder in train_suffixes]

trainfolder, testvolname = trainfolders[0], 'vol7'

ckpts2use = [f'{trainfolder}/{seqname}' for seqname in ['mt', 'cest']]
vol_data_feed_mt, points_grey, points_white = nutils.load_vol_detail(testvolname, seq_name='mt', human_vol_sli=20)
vol_data_feed_amide, _, _ = nutils.load_vol_detail(testvolname, seq_name='amide', human_vol_sli=20)

mt_tissue_params_est, mt_processed_ellipse_d = nutils.infer_ckpt(ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt)
amide_tissue_params_est, amide_processed_ellipse_d = nutils.infer_ckpt(
    ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt, 
    is_cest=True, ckpt_folder_cest=ckpts2use[1], brain2test_cest=vol_data_feed_amide)

In [ ]:
from importlib import reload
reload(nutils)

def infer_and_save_all(do_forward=False):
    for jj, (trainfolder, testvolname) in enumerate(zip(trainfolders, ['vol7', 'vol8', 'vol9', 'vol10'])):
        ckpts2use = [f'{trainfolder}/{seqname}' for seqname in ['mt', 'cest']]
        vol_data_feed_mt, points_grey, points_white = nutils.load_vol_detail(testvolname, seq_name='mt', human_vol_sli=20)
        vol_data_feed_amide, _, _ = nutils.load_vol_detail(testvolname, seq_name='amide', human_vol_sli=20)

        mt_tissue_params_est, mt_processed_ellipse_d = nutils.infer_ckpt(ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt, do_forward=do_forward)
        amide_tissue_params_est, amide_processed_ellipse_d = nutils.infer_ckpt(
            ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt, 
            is_cest=True, ckpt_folder_cest=ckpts2use[1], brain2test_cest=vol_data_feed_amide, do_forward=do_forward)
        np.savez(f'/home/alexf/phiVAE/figs/healthy_apt/{testvolname}/intermediates/apt_tissue_params_est.npz', **amide_tissue_params_est)
        
infer_and_save_all(do_forward=True)

In [ ]:
vol_data_feed_amide.shape


In [ ]:

def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()
mt_sim_mode = 'isar2_c'


def analyze_vs_reference(testvolname, trainfolder, do_all_points = True):
    
    results_folder = f'/home/alexf/phiVAE/figs/healthy_apt/{testvolname}'
    os.makedirs(results_folder, exist_ok=True)
    os.makedirs(results_folder+'/intermediates', exist_ok=True)
    
    vol_data_feed_mt, points_grey, points_white = nutils.load_vol_detail(testvolname, seq_name='mt', human_vol_sli=20)
    vol_data_feed_amide, _, _ = nutils.load_vol_detail(testvolname, seq_name='amide', human_vol_sli=20)
    ckpts2use = [f'{trainfolder}/{seqname}' for seqname in ['mt', 'cest']]
    
    mt_tissue_params_est, mt_processed_ellipse_d = nutils.infer_ckpt(ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt)
    amide_tissue_params_est, amide_processed_ellipse_d = nutils.infer_ckpt(
        ckpt_folder_mt=ckpts2use[0], brain2test_mt=vol_data_feed_mt, 
        is_cest=True, ckpt_folder_cest=ckpts2use[1], brain2test_cest=vol_data_feed_amide)
    
    npz_filename = results_folder+'/intermediates/APT_UQ_stats_allpoints.npz'
    xy_points = meshgrid_xy_pairs(vol_data_feed_amide.shape[0::2])
    if not do_all_points:
        xy_points = xy_points[::10]
        npz_filename = results_folder+'/intermediates/APT_UQ_stats_subset_of_points.npz'   
    print(len(xy_points))    

    mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
        modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
        exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics_dict = \
            au.compare_posteriors(
                xy_points, vol_data_feed_mt, mt_tissue_params_est,
                data_feed_cest=vol_data_feed_amide, cest_tissue_params_est=amide_tissue_params_est, is_cest=True, seq_name='amide', 
                mt_sim_mode=mt_sim_mode, sli=vol_sli, voxels2sample=20000, folder_name=results_folder+'/APT_voxels', visualize=False
            )       

    np.savez_compressed(
        npz_filename,
        mahas1=mahas1, mahas2=mahas2,
        CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
        good_xy_points=good_xy_points, 
        f_est=amide_processed_ellipse_d['f'], k_est=amide_processed_ellipse_d['k'], cov_nnpred_scaled=amide_processed_ellipse_d['cov_nnpred_scaled'],
        covs_of_exact_posterior=covs_of_exact_posterior, modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,
        timings=timings, train_time=0,
        exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
        **extra_metrics_dict
    )
    
RUN_ALL_POINTS_ALL_VOLUMES = False
if RUN_ALL_POINTS_ALL_VOLUMES:
    for jj, (trainfolder, testvolname) in enumerate(zip(trainfolders, ['vol7', 'vol8', 'vol9', 'vol10'])):
        analyze_vs_reference(testvolname, trainfolder)

In [ ]:
def load_all_results(testvolname, do_all_points=True):
    results_folder = f'/home/alexf/phiVAE/figs/healthy_apt/{testvolname}'
    npz_filename = results_folder+f'/intermediates/APT_UQ_stats_{'allpoints' if do_all_points else 'subset_of_points'}.npz'
    data = np.load(npz_filename, allow_pickle=True)
        
    voxel_timings = data['timings']
    print(fr"Bayes per voxel: {np.mean(voxel_timings):.1f}$\pm{np.std(voxel_timings):.1f}$sec")
    
    nn_posterior_min_nrmse_percs = np.mean(data['nn_posterior_min_nrmse_percs'])
    exact_posterior_min_nrmse_percs = np.mean(data['exact_posterior_min_nrmse_percs'])
    
    f_best_dotprod_arr, k_best_dotprod_arr = zip(*data['means_of_exact_posterior'])

    (f_perc_CIs_intersect, k_perc_CIs_intersect, f_perc_NN_CI_covers_exact, k_perc_NN_CI_covers_exact) = \
    au.analyze_CI_intersections(
        data['f_est'], np.array(f_best_dotprod_arr), 
        data['k_est'], np.array(k_best_dotprod_arr), 
        data['cov_nnpred_scaled'], np.array(data['covs_of_exact_posterior']), data['good_xy_points'], 
        sli=vol_sli, is_mt=False, figsize=(3, 3), df=0.008, dk=15, fmin=0.07, fmax=0.5
    )
    plt.show()
    
    mahas1 = np.median(data['mahas1'].reshape(-1, 10), axis=1)
    mahas2 = np.median(data['mahas2'].reshape(-1, 10), axis=1)
    # after taking voxelwise typical, find outliers % and typical across voxels
    mahas1_percBAD = 100*np.sum(mahas1>4)/len(mahas1)
    mahas2_percBAD = 100*np.sum(mahas2>4)/len(mahas2)
    mahas1_typical = np.median(mahas1)
    mahas2_typical = np.median(mahas2)

    humanAPT_KLdist_median = np.median(data['KL(P1||P2)'])
    humanAPT_JS_median = np.median(data['Jensen-Shannon Divergence'])
    humanAPT_Hell_median = np.median(data['Hellinger Distance'])       
    
    return {'nn_posterior_min_nrmse_percs': nn_posterior_min_nrmse_percs,
            'exact_posterior_min_nrmse_percs': exact_posterior_min_nrmse_percs,
            'f_perc_CIs_intersect': f_perc_CIs_intersect,
            'k_perc_CIs_intersect': k_perc_CIs_intersect,
            'mahas1_percBAD': mahas1_percBAD,
            'mahas2_percBAD': mahas2_percBAD,
            'mahas1_typical': mahas1_typical,
            'mahas2_typical': mahas2_typical,
            'raw_mahas1': data['mahas1'],
            'raw_mahas2': data['mahas2'],
            'humanAPT_KLdist_median': humanAPT_KLdist_median,
            'humanAPT_JS_median': humanAPT_JS_median,
            'humanAPT_Hell_median': humanAPT_Hell_median,
            #'JS_percBAD': JS_percBAD
            }

results = [load_all_results(vol, do_all_points=True) for vol in ['vol7', 'vol8', 'vol9', 'vol10']]


In [ ]:
results[0]

In [ ]:
metric_lists = {metric: [result[metric] for result in results] for metric in results[0].keys()}

import pandas as pd
df = pd.DataFrame()
df['Healthy\\ Humans,\\ APT\\ 3T\\ pulsed'] = [         
        f'NN: {np.mean(metric_lists["nn_posterior_min_nrmse_percs"]):.1f}\\%'+'$\\pm$'+f'{np.std(metric_lists["nn_posterior_min_nrmse_percs"]):.1f}\\% ',
        f'Ref: {np.mean(metric_lists["exact_posterior_min_nrmse_percs"]):.1f}\\%'+'$\\pm$'+f'{np.std(metric_lists["exact_posterior_min_nrmse_percs"]):.2f}\\% ',
        # # est correlation
        ' ',' ',
        # CI-size correlation
        ' ',' ',
        # # CI intersect
        'f$_{s}$:\\ '+f'{np.mean(metric_lists["f_perc_CIs_intersect"]):.1f}\\%'+'$\\pm$'+f'{np.std(metric_lists["f_perc_CIs_intersect"]):.1f}\\%',
        'k$_{sw}$:\\ '+f'{np.mean(metric_lists["k_perc_CIs_intersect"]):.1f}\\%'+'$\\pm$'+f'{np.std(metric_lists["k_perc_CIs_intersect"]):.1f}\\%',
        # Mahalanobises
        f"M$_1$={np.nanmean(metric_lists['mahas1_typical']):.2f}$\\pm${np.nanstd(metric_lists['mahas1_typical']):.2f}", 
        f"M$_2$={np.nanmean(metric_lists['mahas2_typical']):.2f}$\\pm${np.nanstd(metric_lists['mahas2_typical']):.2f}", 
        f"$M_1>4$:\\ {np.mean(metric_lists['mahas1_percBAD']):.1f}\\%$\\pm${np.std(metric_lists['mahas1_percBAD']):.1f}\\% (n={len(metric_lists['mahas1_percBAD'])})",
        f"$M_2>4$:\\ {np.mean(metric_lists['mahas2_percBAD']):.1f}\\%$\\pm${np.std(metric_lists['mahas2_percBAD']):.1f}\\% (n={len(metric_lists['mahas2_percBAD'])})",
        # Dist divergences
        f"KL:\\ {np.mean(metric_lists['humanAPT_KLdist_median']):.2f}$\\pm${np.std(metric_lists['humanAPT_KLdist_median']):.2f}", 
        f"JS:\\ {np.mean(metric_lists['humanAPT_JS_median']):.2f}$\\pm${np.std(metric_lists['humanAPT_JS_median']):.2f}",
        f"HL:\\ {np.mean(metric_lists['humanAPT_Hell_median']):.2f}$\\pm${np.std(metric_lists['humanAPT_Hell_median']):.2f}",
        ]

print(df)
df.to_csv('/home/alexf/phiVAE/figs/healthy_apt/human_APT_summary.csv', index=True)

## Create and save all voxel PDFs for a slice

In [ ]:
# _df, _dk (0.003, 0.7) 

In [ ]:
def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()
mt_sim_mode = 'isar2_c'

def get_all_pdfs(data_feed_mt, sli, seq_name='mt', mt_sim_mode='isar2_c', **kwargs):
    #_x, _y, sli 
    good_xy_points = []
    allpdfs = []
    min_nrms_list, min_nrms_95cr_th_list, CR_areas = [], [], []
    xy_points = meshgrid_xy_pairs(data_feed_mt.shape[0::2])
    try: 
        for _x, _y in xy_points:
            if np.isnan(data_feed_mt.roi_mask_nans[_x, sli, _y]):
                continue   
            good_xy_points.append((_x, _y))
            (f_best,k_best, _, nrmse, _df, _dk,) = au.core.get_nrmse_grid(
                None, _x, _y, sli, data_feed_mt,
                seq_name=seq_name, mt_sim_mode=mt_sim_mode,
                **kwargs
                )
            (
                pdf, nrmse_95cr_th, CR_area,
                f_mean_grid_posterior, f_025, f_975,
                k_mean_grid_posterior, k_025, k_975,
                posterior_cov, cdf_at_gt
            ) = au.core.nrmse_to_CIs(nrmse, is_cest=(seq_name!='mt'), df=_df, dk=_dk, gt=None)
            allpdfs.append(pdf)
            min_nrms = np.min(nrmse)
            min_nrms_list.append(min_nrms)
            min_nrms_95cr_th_list.append(nrmse_95cr_th)
            CR_areas.append(CR_area)
    except KeyboardInterrupt as e:
        print("Interrupted by user..")
    print("Processed {} voxels.".format(len(good_xy_points)))
        
    return allpdfs, good_xy_points, min_nrms_list, min_nrms_95cr_th_list, CR_areas

for testvolname in ['vol8', 'vol9', 'vol10','vol7']:
    vol_data_feed_mt, points_grey, points_white = nutils.load_vol_detail(testvolname, seq_name='mt')
    allpdfs, good_xy_points, min_nrms_list, min_nrms_95cr_th_list, CR_areas = get_all_pdfs(vol_data_feed_mt, sli=15, seq_name='mt', mt_sim_mode=mt_sim_mode)
    np.savez_compressed(
        f'/home/alexf/phiVAE/figs/humans/vol_pdfs/{testvolname}.npz', 
        allpdfs=allpdfs, good_xy_points=good_xy_points, 
        min_nrms_list=min_nrms_list, min_nrms_95cr_th_list=min_nrms_95cr_th_list, CR_areas=CR_areas
    )